[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/main/notebooks/patterns/crewai_patterns.ipynb)

# Seven matched patterns with CrewAI

**Task.** Answer *Which interventions reduce household food waste, and under what conditions?* while comparing seven CrewAI-native orchestration patterns.

The bounded catalogue contains three tutorial records: smaller plates, meal planning, and an information-only campaign. Each record has a source ID, title, topics and evidence text. All framework notebooks use these records and the same versioned mock decisions, so the comparison focuses on orchestration.

**Read this notebook as:** setup → shared contracts/adapters → native Agent/Task/Crew examples → matched evaluation. Runtime is about one minute on CPU; no key or model download is required.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "crewai==1.15.2", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import asyncio
import contextlib
import io
import json
import os
import tempfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any, ClassVar

import appdirs

os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
appdirs.user_data_dir = lambda *args, **kwargs: str(
    Path(tempfile.gettempdir()) / "agentic-ai-tutorial-crewai"
)

from crewai import Agent, BaseLLM, Crew, Process, Task  # noqa: E402
from crewai.flow.flow import Flow, listen, router, start  # noqa: E402
from crewai.tasks.conditional_task import ConditionalTask  # noqa: E402
from crewai.tools import tool  # noqa: E402
from pydantic import BaseModel, ConfigDict, Field, PrivateAttr  # noqa: E402

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))

from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import (  # noqa: E402
    CriticDecision,
    Message,
    MessageRole,
    PlanDecision,
    RouteDecision,
    ToolDefinition,
)

fixture_data = json.loads(
    (ROOT / "data/research_assistant/pattern_mock_scenarios.json").read_text()
)
catalogue = fixture_data["catalogue"]
TASK = "Which interventions reduce household food waste, and under what conditions?"

## What comes from where

- **CrewAI:** `Agent` defines a role/goal, `Task` defines work and typed output, `Crew` runs collaborating agents, `ConditionalTask` gates revision, `@tool` exposes bounded actions, and `Flow` provides event routing.
- **Pydantic:** validates structured decisions and final answers.
- **Repository:** the deterministic model client, versioned fixture scenarios, shared route/plan/critic schemas, messages and tool contract.
- **This notebook:** evidence schemas, a thin `FixtureCrewLLM` shape adapter, the catalogue tool, a deterministic worker adapter, crews and evaluation checks.

The adapter translates shared fixture responses into CrewAI's LLM interface; it does not choose routes, schedule tasks or execute tools. Those responsibilities remain with CrewAI abstractions. Flow is used only for the routing example, where event dispatch is the native abstraction.

In [ ]:
# --- Notebook-defined contracts, fixture adapter and bounded tool ---
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Claim(StrictModel):
    schema_id: ClassVar[str] = "tutorial.claim.v1"
    claim: str
    source_id: str
    supported: bool


class Answer(StrictModel):
    schema_id: ClassVar[str] = "tutorial.pattern_answer.v1"
    answer: str
    source_ids: tuple[str, ...]


class ParallelDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.parallel.v1"
    queries: tuple[str, ...] = Field(min_length=2, max_length=3)
    aggregation: str = Field(pattern="^validated_union$")


class SearchBatch(StrictModel):
    worker: str
    records: tuple[dict, ...]


class WorkerAssignment(StrictModel):
    worker: str = Field(pattern="^(intervention_reviewer|planning_reviewer)$")
    query: str


class OrchestrationDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.orchestration.v1"
    assignments: tuple[WorkerAssignment, ...] = Field(min_length=1, max_length=2)


OrchestrationDecision.model_rebuild(_types_namespace={"WorkerAssignment": WorkerAssignment})


def model_for(name: str) -> DeterministicMockClient:
    fixture = ScriptedScenarioFixture.model_validate(
        {
            "fixture_version": fixture_data["fixture_version"],
            "scenario": name,
            "provenance": fixture_data["provenance"],
            "responses": fixture_data["scenarios"][name],
        }
    )
    return DeterministicMockClient(fixture)


def user(text: str) -> Message:
    return Message(role=MessageRole.USER, content=text)


def search(query: str, limit: int = 3) -> list[dict]:
    terms = set(query.casefold().split())
    return [
        record for record in catalogue if terms & set(" ".join(record["topics"]).casefold().split())
    ][:limit]


@tool("search_catalogue")
def search_catalogue(query: str, max_results: int) -> str:
    """Search the bounded food-waste catalogue."""
    return json.dumps(search(query, max_results))


search_contract = ToolDefinition(
    name="search_catalogue",
    description="Search the bounded food-waste catalogue.",
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string"}, "max_results": {"type": "integer"}},
        "required": ["query", "max_results"],
    },
)


def resolve(coroutine):
    """Run the async shared client safely behind CrewAI's synchronous LLM interface."""
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(asyncio.run, coroutine).result()


class FixtureCrewLLM(BaseLLM):
    """Translate one versioned scenario into CrewAI LLM responses."""

    _core: Any = PrivateAttr()
    _scenario: str = PrivateAttr()

    def __init__(self, scenario: str):
        super().__init__(model="patterns-mock-v1", provider="deterministic-mock")
        self._scenario = scenario
        self._core = model_for(scenario)

    def call(
        self,
        messages,
        tools=None,
        callbacks=None,
        available_functions=None,
        from_task=None,
        from_agent=None,
        response_model=None,
    ):
        use_native_tool = self._scenario == "react" and from_agent and from_agent.tools
        response = resolve(
            self._core.generate(
                [user(str(messages))],
                response_schema=None if use_native_tool else response_model,
                tools=[search_contract] if use_native_tool else (),
            )
        )
        if response.tool_calls:
            call = response.tool_calls[0]
            return (
                "Thought: Search the bounded catalogue.\n"
                f"Action: {call.name}\n"
                f"Action Input: {json.dumps(call.arguments)}"
            )
        return json.dumps(response.structured_output, sort_keys=True)


class CatalogueWorkerLLM(BaseLLM):
    """Deterministic worker model used to demonstrate native async CrewAI tasks."""

    _worker: str = PrivateAttr()
    _query: str = PrivateAttr()
    _calls: int = PrivateAttr(default=0)

    def __init__(self, worker: str, query: str):
        super().__init__(model="catalogue-worker-v1", provider="deterministic-local")
        self._worker = worker
        self._query = query

    def call(
        self,
        messages,
        tools=None,
        callbacks=None,
        available_functions=None,
        from_task=None,
        from_agent=None,
        response_model=None,
    ):
        self._calls += 1
        if self._calls == 1:
            arguments = {"query": self._query, "max_results": 2}
            return (
                "Thought: Use the read-only catalogue tool.\n"
                "Action: search_catalogue\n"
                f"Action Input: {json.dumps(arguments)}"
            )
        return SearchBatch(
            worker=self._worker,
            records=tuple(search(self._query, 2)),
        ).model_dump_json()


class AggregateLLM(BaseLLM):
    """Return a typed summary after CrewAI has awaited all worker tasks."""

    def __init__(self):
        super().__init__(model="aggregate-v1", provider="deterministic-local")

    def call(
        self,
        messages,
        tools=None,
        callbacks=None,
        available_functions=None,
        from_task=None,
        from_agent=None,
        response_model=None,
    ):
        return Answer(
            answer="The worker evidence is ready for evaluation.",
            source_ids=("food-waste-001", "food-waste-002"),
        ).model_dump_json()


async def quiet_kickoff(crew: Crew):
    """Hide decorative panels only; CrewAI still executes every agent/task/tool."""
    with contextlib.redirect_stdout(io.StringIO()):
        return await crew.kickoff_async()

## 1–2. Prompt chaining and routing

Prompt chaining uses two sequential typed tasks with task context. Routing uses a one-task triage crew inside a small `Flow`; `@router` and `@listen` perform the actual dispatch.

In [ ]:
# --- 1. Prompt chaining: sequential Task context ---
chain_llm = FixtureCrewLLM("prompt_chaining")
extractor = Agent(
    role="Evidence extractor",
    goal="Extract one supported, cited claim",
    backstory="A careful evidence reviewer",
    llm=chain_llm,
    verbose=False,
)
writer = Agent(
    role="Evidence writer",
    goal="Synthesis only validated claims",
    backstory="A concise research writer",
    llm=chain_llm,
    verbose=False,
)
extract_task = Task(
    name="extract",
    description="Extract the supported food-waste-001 claim.",
    expected_output="A Claim JSON object",
    agent=extractor,
    output_pydantic=Claim,
)
synthesise_task = Task(
    name="synthesise",
    description="Synthesis only the claim supplied in task context.",
    expected_output="An Answer JSON object",
    agent=writer,
    context=[extract_task],
    output_pydantic=Answer,
)
chain_crew = Crew(
    agents=[extractor, writer],
    tasks=[extract_task, synthesise_task],
    process=Process.sequential,
    verbose=False,
)
await quiet_kickoff(chain_crew)
chain_state = {
    "answer": synthesise_task.output.pydantic.model_dump(),
    "trace": ["Task:extract", "context", "Task:synthesise"],
    "stop": "criteria_met",
}


# --- 2. Routing: a triage Crew produces state; Flow routes it ---
class RoutingFlow(Flow):
    @start()
    async def triage(self):
        llm = FixtureCrewLLM("routing")
        triage_agent = Agent(
            role="Triage agent",
            goal="Select research or clarify",
            backstory="A bounded request classifier",
            llm=llm,
            verbose=False,
        )
        triage_task = Task(
            name="triage",
            description=f"Route this request: {TASK}",
            expected_output="A RouteDecision JSON object",
            agent=triage_agent,
            output_pydantic=RouteDecision,
        )
        decision = (
            await quiet_kickoff(Crew(agents=[triage_agent], tasks=[triage_task], verbose=False))
        ).pydantic
        self.state["decision"] = decision.model_dump()
        return decision.route

    @router(triage)
    def dispatch(self, route):
        return route if route in {"research", "clarify"} else "clarify"

    @listen("research")
    def handle_research(self):
        return {
            "results": search("meal planning portion"),
            "trace": ["Task:triage", "Flow:@router", "Flow:research"],
            "stop": "criteria_met",
        }

    @listen("clarify")
    def handle_clarify(self):
        return {
            "results": [],
            "trace": ["Task:triage", "Flow:@router", "Flow:clarify"],
            "stop": "safe_fallback",
        }


with contextlib.redirect_stdout(io.StringIO()):
    route_state = await RoutingFlow().kickoff_async()

## 3–5. Parallelisation, ReAct and planner–executor

Parallel work uses CrewAI's `async_execution=True` tasks. ReAct gives an agent a real `@tool`; CrewAI parses the action, executes the tool and returns the observation to the agent. Planning uses a typed planning task before deterministic dependency execution.

In [ ]:
# --- 3. Parallelisation: native async tasks in one Crew ---
parallel_planner_llm = FixtureCrewLLM("parallelisation")
parallel_planner = Agent(
    role="Parallel planner",
    goal="Propose independent bounded searches",
    backstory="A research coordinator",
    llm=parallel_planner_llm,
    verbose=False,
)
parallel_plan_task = Task(
    name="parallel_plan",
    description=f"Propose two independent searches for: {TASK}",
    expected_output="A ParallelDecision JSON object",
    agent=parallel_planner,
    output_pydantic=ParallelDecision,
)
parallel_decision = (
    await quiet_kickoff(Crew(agents=[parallel_planner], tasks=[parallel_plan_task], verbose=False))
).pydantic

parallel_agents, parallel_tasks = [], []
for index, query in enumerate(parallel_decision.queries, start=1):
    worker_name = f"parallel_worker_{index}"
    worker_agent = Agent(
        role=worker_name,
        goal="Search one assigned topic",
        backstory="A bounded catalogue worker",
        llm=CatalogueWorkerLLM(worker_name, query),
        tools=[search_catalogue],
        max_iter=3,
        verbose=False,
    )
    worker_task = Task(
        name=worker_name,
        description=f"Search only for: {query}",
        expected_output="A SearchBatch JSON object",
        agent=worker_agent,
        output_pydantic=SearchBatch,
        async_execution=True,
    )
    parallel_agents.append(worker_agent)
    parallel_tasks.append(worker_task)

parallel_aggregator = Agent(
    role="Evidence aggregator",
    goal="Aggregate completed worker evidence",
    backstory="A deterministic evidence collector",
    llm=AggregateLLM(),
    verbose=False,
)
parallel_aggregate_task = Task(
    name="parallel_aggregate",
    description="Aggregate the completed worker batches supplied in context.",
    expected_output="An Answer JSON object",
    agent=parallel_aggregator,
    context=parallel_tasks,
    output_pydantic=Answer,
)
parallel_crew = Crew(
    agents=[*parallel_agents, parallel_aggregator],
    tasks=[*parallel_tasks, parallel_aggregate_task],
    process=Process.sequential,
    verbose=False,
)
with contextlib.redirect_stdout(io.StringIO()):
    await parallel_crew.kickoff_async()
parallel_batches = [task.output.pydantic for task in parallel_tasks]
parallel_source_ids = sorted(
    {record["source_id"] for batch in parallel_batches for record in batch.records}
)
parallel_state = {
    "source_ids": parallel_source_ids,
    "trace": ["Task:parallel_plan", "async Tasks", "validated_union"],
    "stop": "criteria_met",
}


# --- 4. ReAct: Agent + native tool execution ---
react_llm = FixtureCrewLLM("react")
react_agent = Agent(
    role="ReAct researcher",
    goal="Answer using the bounded catalogue tool",
    backstory="A tool-using evidence researcher",
    llm=react_llm,
    tools=[search_catalogue],
    max_iter=3,
    verbose=False,
)
react_task = Task(
    name="react",
    description=f"Use search_catalogue, then answer: {TASK}",
    expected_output="An Answer JSON object",
    agent=react_agent,
    output_pydantic=Answer,
)
react_answer = (
    await quiet_kickoff(Crew(agents=[react_agent], tasks=[react_task], verbose=False))
).pydantic
react_state = {
    "answer": react_answer.model_dump(),
    "trace": ["Agent action", "@tool execution", "Agent finish"],
    "steps": 2,
    "stop": "criteria_met",
}


# --- 5. Planner-executor: typed Task followed by dependency gate ---
planner_llm = FixtureCrewLLM("planner_executor")
planner_agent = Agent(
    role="Planner",
    goal="Create a dependency-aware research plan",
    backstory="A cautious workflow planner",
    llm=planner_llm,
    verbose=False,
)
planner_task = Task(
    name="plan",
    description=f"Create a three-step plan for: {TASK}",
    expected_output="A PlanDecision JSON object",
    agent=planner_agent,
    output_pydantic=PlanDecision,
)
plan = (
    await quiet_kickoff(Crew(agents=[planner_agent], tasks=[planner_task], verbose=False))
).pydantic
plan_valid = len(plan.steps) == 3 and all(
    plan.steps[index].depends_on == (plan.steps[index - 1].step_id,) for index in (1, 2)
)
plan_state = {
    "plan": plan.model_dump(),
    "trace": ["Task:plan", "dependency_executor" if plan_valid else "dependency_stop"],
    "stop": "criteria_met" if plan_valid else "invalid_plan",
}

## 6–7. Critic–reviser and orchestrator–worker

The critic crew uses task context plus `ConditionalTask`, so revision runs only after rejection. The orchestrator first creates typed assignments, then launches native async worker tasks and gives their outputs to a synthesis task.

In [ ]:
# --- 6. Critic-reviser: context + ConditionalTask ---
critic_llm = FixtureCrewLLM("critic_reviser")
draft_agent = Agent(
    role="Draft writer",
    goal="Draft an evidence answer",
    backstory="An initial research writer",
    llm=critic_llm,
    verbose=False,
)
critic_agent = Agent(
    role="Critic",
    goal="Reject unsupported or unconditional claims",
    backstory="An independent evidence critic",
    llm=critic_llm,
    verbose=False,
)
reviser_agent = Agent(
    role="Reviser",
    goal="Repair the draft once",
    backstory="A bounded revision specialist",
    llm=critic_llm,
    verbose=False,
)
draft_task = Task(
    name="draft",
    description="Draft the answer.",
    expected_output="An Answer JSON object",
    agent=draft_agent,
    output_pydantic=Answer,
)
critique_task = Task(
    name="critique",
    description="Check the draft supplied in context.",
    expected_output="A CriticDecision JSON object",
    agent=critic_agent,
    context=[draft_task],
    output_pydantic=CriticDecision,
)
revision_task = ConditionalTask(
    name="revise",
    description="Revise once using the draft and critique in context.",
    expected_output="A corrected Answer JSON object",
    agent=reviser_agent,
    context=[draft_task, critique_task],
    output_pydantic=Answer,
    condition=lambda output: bool(output.pydantic is not None and not output.pydantic.accepted),
)
critic_crew = Crew(
    agents=[draft_agent, critic_agent, reviser_agent],
    tasks=[draft_task, critique_task, revision_task],
    process=Process.sequential,
    verbose=False,
)
await quiet_kickoff(critic_crew)
critic_state = {
    "answer": revision_task.output.pydantic.model_dump(),
    "revisions": 1,
    "trace": ["Task:draft", "Task:critique", "ConditionalTask:revise"],
    "stop": "criteria_met",
}


# --- 7. Orchestrator-worker: typed assignment + native async Tasks ---
orchestrator_llm = FixtureCrewLLM("orchestrator_worker")
orchestrator_agent = Agent(
    role="Orchestrator",
    goal="Assign bounded specialist searches",
    backstory="A delegation coordinator",
    llm=orchestrator_llm,
    verbose=False,
)
assignment_task = Task(
    name="assign",
    description=f"Assign two named reviewers for: {TASK}",
    expected_output="An OrchestrationDecision JSON object",
    agent=orchestrator_agent,
    output_pydantic=OrchestrationDecision,
)
assignments = (
    await quiet_kickoff(Crew(agents=[orchestrator_agent], tasks=[assignment_task], verbose=False))
).pydantic

worker_agents, worker_tasks = [], []
for assignment in assignments.assignments:
    worker_agent = Agent(
        role=assignment.worker,
        goal="Complete one assigned catalogue search",
        backstory="A bounded evidence specialist",
        llm=CatalogueWorkerLLM(assignment.worker, assignment.query),
        tools=[search_catalogue],
        max_iter=3,
        verbose=False,
    )
    worker_task = Task(
        name=assignment.worker,
        description=f"Search only for: {assignment.query}",
        expected_output="A SearchBatch JSON object",
        agent=worker_agent,
        output_pydantic=SearchBatch,
        async_execution=True,
    )
    worker_agents.append(worker_agent)
    worker_tasks.append(worker_task)

synthesis_agent = Agent(
    role="Synthesis agent",
    goal="Combine only worker evidence",
    backstory="A source-grounded research writer",
    llm=orchestrator_llm,
    verbose=False,
)
synthesis_task = Task(
    name="worker_synthesis",
    description="Synthesise only the worker batches supplied in task context.",
    expected_output="An Answer JSON object",
    agent=synthesis_agent,
    context=worker_tasks,
    output_pydantic=Answer,
)
worker_crew = Crew(
    agents=[*worker_agents, synthesis_agent],
    tasks=[*worker_tasks, synthesis_task],
    process=Process.sequential,
    verbose=False,
)
worker_answer = (await quiet_kickoff(worker_crew)).pydantic
worker_batches = [task.output.pydantic for task in worker_tasks]
orchestrator_state = {
    "answer": worker_answer.model_dump(),
    "worker_count": len(worker_batches),
    "trace": ["Task:assign", "async worker Tasks", "Task:worker_synthesis"],
    "stop": "criteria_met",
}

## Evaluation summary

Each pattern must terminate and meet the same observable check used by the other framework notebooks.

Limitations: deterministic adapters exercise CrewAI orchestration rather than hosted-model judgment; task retries and tool loops still need production budgets; and native crews add more runtime and dependency overhead than the plain-Python baseline.

In [ ]:
pattern_evaluations = {
    "prompt_chaining": chain_state["stop"] == "criteria_met",
    "routing": route_state["stop"] == "criteria_met" and len(route_state["results"]) == 2,
    "parallelisation": parallel_state["source_ids"] == ["food-waste-001", "food-waste-002"],
    "react": react_state["stop"] == "criteria_met" and react_state["steps"] <= 2,
    "planner_executor": plan_state["stop"] == "criteria_met",
    "critic_reviser": critic_state["revisions"] == 1,
    "orchestrator_worker": orchestrator_state["worker_count"] == 2,
}
pattern_limitations = {
    "prompt_chaining": "early errors propagate",
    "routing": "valid routes can still be semantically wrong",
    "parallelisation": "async tasks can duplicate work",
    "react": "tool loops require strict budgets",
    "planner_executor": "invalid dependencies stop execution",
    "critic_reviser": "one revision may be insufficient",
    "orchestrator_worker": "worker findings can conflict",
}
assert set(pattern_limitations) == set(pattern_evaluations)
assert all(pattern_evaluations.values()), pattern_evaluations
{
    "framework": "CrewAI",
    "evaluation": pattern_evaluations,
    "native_abstractions": [
        "Agent",
        "Task",
        "Crew",
        "ConditionalTask",
        "async_execution",
        "@tool",
        "Flow routing",
    ],
    "limitations": pattern_limitations,
}